# Transformer Internals for Inference

This note goes deeper than basic attention — focused on *inference-time* concerns: KV caching, RoPE, grouped-query attention, and sampling strategies. These are the topics LLM-track MLE interviews probe after you've shown you know basic attention.

## What Interviewers Test
- KV cache: what it stores, why it speeds up decoding, memory cost
- RoPE vs sinusoidal positional encoding — what changes at long contexts
- GQA: how it reduces KV cache memory without losing much quality
- Sampling strategies: greedy, temperature, top-k, top-p — mathematical implementation
- Tokenizer effects on cost (tokens ≠ characters)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# ===== KV Cache Implementation =====
def scaled_dot_product_attention(Q, K, V, mask=None):
    B, T_q, d_k = Q.shape
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
    if mask is not None:
        scores += mask * -1e9
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights @ V

class AttentionWithKVCache:
    def __init__(self, d_model, n_heads):
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        scale = np.sqrt(2 / d_model)
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
        # KV Cache
        self.cache_K = None
        self.cache_V = None

    def forward_prefill(self, X):
        """Process full prompt; populate KV cache."""
        B, T, _ = X.shape
        Q = (X @ self.W_Q).reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        K = (X @ self.W_K).reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        V = (X @ self.W_V).reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        self.cache_K = K  # (B, H, T, d_k)
        self.cache_V = V  # (B, H, T, d_k)
        # Full causal attention
        B, H, T, d_k = Q.shape
        out = scaled_dot_product_attention(Q.reshape(B*H, T, d_k),
                                           K.reshape(B*H, T, d_k),
                                           V.reshape(B*H, T, d_k))
        out = out.reshape(B, H, T, d_k).transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
        return out @ self.W_O

    def forward_decode(self, x_new):
        """Process one new token; use cached K,V."""
        B, _, _ = x_new.shape   # x_new: (B, 1, d_model)
        q = (x_new @ self.W_Q).reshape(B, 1, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        k = (x_new @ self.W_K).reshape(B, 1, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        v = (x_new @ self.W_V).reshape(B, 1, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        # Append to cache
        self.cache_K = np.concatenate([self.cache_K, k], axis=2)
        self.cache_V = np.concatenate([self.cache_V, v], axis=2)
        # Attend over all cached tokens
        B, H, T_cache, d_k = self.cache_K.shape
        out = scaled_dot_product_attention(q.reshape(B*H, 1, d_k),
                                           self.cache_K.reshape(B*H, T_cache, d_k),
                                           self.cache_V.reshape(B*H, T_cache, d_k))
        out = out.reshape(B, H, 1, d_k).transpose(0, 2, 1, 3).reshape(B, 1, self.d_model)
        return out @ self.W_O

# Benchmark: with vs without KV cache
B, d_model, n_heads, prompt_len = 1, 64, 4, 32
attn = AttentionWithKVCache(d_model, n_heads)
prompt = np.random.randn(B, prompt_len, d_model)

import time
# With cache
t0 = time.time()
_ = attn.forward_prefill(prompt)
for _ in range(20):
    new_tok = np.random.randn(B, 1, d_model)
    _ = attn.forward_decode(new_tok)
t_cache = time.time() - t0

print(f"With KV cache (20 decode steps): {t_cache*1000:.1f}ms")
print(f"Cache holds {attn.cache_K.shape[2]} key/value pairs after generation")

# Memory of KV cache
n_layers, bytes_per_element = 12, 2  # fp16
kv_mem = 2 * n_layers * B * n_heads * (prompt_len + 20) * (d_model // n_heads) * bytes_per_element
print(f"KV cache memory: {kv_mem} bytes (scales linearly with sequence length)")


## KV Cache Memory Formula

For a model with $L$ layers, $H$ heads, sequence length $T$, head dimension $d_k$, batch $B$:

$$\text{KV cache} = 2 \times L \times B \times H \times T \times d_k \times \text{bytes}$$

For LLaMA-2 7B (L=32, H=32, d_k=128) at 4K context, fp16:
$$= 2 \times 32 \times 1 \times 32 \times 4096 \times 128 \times 2 \approx 2.1 \text{ GB}$$

**Grouped-Query Attention (GQA):** Use fewer KV heads ($H_{kv} < H$) while keeping full Q heads. If $H_{kv} = 8$ vs $H = 32$: 4× KV cache reduction with minimal quality loss. Used in LLaMA-2 70B, Mistral, Gemma.


In [ ]:
# ===== RoPE vs Sinusoidal =====
def sinusoidal_pe(T, d):
    pos = np.arange(T)[:, None]
    dims = np.arange(0, d, 2)[None, :]
    PE = np.zeros((T, d))
    PE[:, 0::2] = np.sin(pos / 10000**(dims/d))
    PE[:, 1::2] = np.cos(pos / 10000**(dims/d))
    return PE

def rope_rotate(x, cos, sin):
    """Apply RoPE rotation to (B, H, T, d_k) tensor."""
    x1, x2 = x[..., ::2], x[..., 1::2]
    return np.concatenate([x1*cos - x2*sin, x1*sin + x2*cos], axis=-1)

def build_rope_freqs(T, d_k, base=10000):
    theta = 1.0 / base**(np.arange(0, d_k, 2) / d_k)  # (d_k/2,)
    t = np.arange(T)
    freqs = np.outer(t, theta)    # (T, d_k/2)
    cos = np.cos(freqs)           # (T, d_k/2)
    sin = np.sin(freqs)
    return cos, sin

# Key property: RoPE encodes RELATIVE position, not absolute
T, d_k = 20, 32
cos, sin = build_rope_freqs(T, d_k)
print("RoPE key property:")
print(f"cos shape: {cos.shape}  (T, d_k/2)")
print("RoPE rotates query/key vectors — inner product between rotated q and k")
print("depends only on their RELATIVE position (i - j), not absolute positions.")
print("This enables better extrapolation to longer contexts than sinusoidal.")


In [ ]:
# ===== Sampling Strategies =====
def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def sample_greedy(logits):
    """Always pick most probable token."""
    return int(logits.argmax())

def sample_temperature(logits, temperature=1.0):
    """Scale logits by temperature before sampling."""
    scaled = logits / temperature
    probs = softmax(scaled)
    return int(np.random.choice(len(probs), p=probs))

def sample_top_k(logits, k=50, temperature=1.0):
    """Sample from top-k tokens only."""
    scaled = logits / temperature
    top_k_idx = np.argpartition(scaled, -k)[-k:]
    top_k_logits = np.full_like(scaled, -np.inf)
    top_k_logits[top_k_idx] = scaled[top_k_idx]
    probs = softmax(top_k_logits)
    return int(np.random.choice(len(probs), p=probs))

def sample_top_p(logits, p=0.9, temperature=1.0):
    """Nucleus sampling: sample from smallest set whose prob mass >= p."""
    scaled = logits / temperature
    probs = softmax(scaled)
    sorted_idx = np.argsort(-probs)
    cum_probs = np.cumsum(probs[sorted_idx])
    cutoff = np.searchsorted(cum_probs, p) + 1
    nucleus_idx = sorted_idx[:cutoff]
    nucleus_probs = probs[nucleus_idx]
    nucleus_probs /= nucleus_probs.sum()
    return int(nucleus_idx[np.random.choice(len(nucleus_probs), p=nucleus_probs)])

# Demonstrate on a simple vocabulary
vocab_size = 20
np.random.seed(7)
logits = np.random.randn(vocab_size) * 2

print("=== Sampling Strategy Comparison ===")
np.random.seed(42)
for temp in [0.5, 1.0, 2.0]:
    samples = [sample_temperature(logits, temp) for _ in range(1000)]
    unique = len(set(samples))
    top_token_pct = max(np.bincount(samples)) / 1000
    print(f"Temperature={temp}: {unique} unique tokens, top token={top_token_pct:.1%} of samples")

print()
samples_topk  = [sample_top_k(logits, k=5)  for _ in range(1000)]
samples_topp  = [sample_top_p(logits, p=0.9) for _ in range(1000)]
print(f"Top-k=5:  {len(set(samples_topk))} unique tokens")
print(f"Top-p=0.9: {len(set(samples_topp))} unique tokens")


## Common Interview Questions

**Q: What is the KV cache and why does it make decoding efficient?**
During generation, each new token needs to attend to all previous tokens. Without caching, we'd recompute K and V for every past token at every step — O(T²) total. The KV cache stores K and V for past tokens so each decode step only needs to compute Q, K, V for the new token and attend to cached past K, V — reducing decode from O(T²) to O(T) total.

**Q: Why use temperature in sampling?**
Temperature divides logits before softmax. Low temperature (<1) sharpens the distribution toward the top token — more deterministic, less creative. High temperature (>1) flattens the distribution — more diverse but potentially incoherent. Temperature=0 is equivalent to greedy. For factual generation, use low temperature; for creative tasks, use higher temperature.

**Q: What is top-p (nucleus) sampling and when is it preferred over top-k?**
Top-k always samples from exactly k tokens regardless of the distribution. If the model is confident (peaked distribution), k=50 still includes many low-probability tokens. Top-p adapts the vocabulary size: it includes the smallest set of tokens whose cumulative probability exceeds p. When the model is confident, nucleus is small; when uncertain, nucleus is large. This makes top-p more robust across different confidence levels.

**Q: Why does GQA reduce KV cache memory?**
Standard multi-head attention has H query heads, H key heads, and H value heads. GQA uses fewer KV heads (e.g., 8 instead of 32) that are shared across groups of query heads. The KV cache only stores H_kv key/value matrices instead of H, reducing cache size by H/H_kv (4× for 32→8 heads), enabling longer contexts or larger batch sizes.

## Key Takeaways
- KV cache: store past K and V to avoid recomputation; memory scales as 2×L×B×H×T×d_k×bytes
- Prefill is parallelizable (process entire prompt at once); decode is sequential (one token at a time)
- RoPE encodes relative position via rotation — better extrapolation to long contexts than sinusoidal
- GQA: fewer KV heads than Q heads → proportional KV cache reduction (LLaMA-2 70B, Mistral use it)
- Sampling: greedy (deterministic) → temperature (scales diversity) → top-k → top-p (adaptive nucleus)
- Temperature < 1: more focused; > 1: more random; top-p adapts vocabulary size to model confidence